# Symmetric Bipodal Versus Broader Families

This notebook compares the symmetric-bipodal baseline with unrestricted bipodal and higher-podal candidates in the saved target comparisons and coarse entropy grids. It also renders pode heatmaps for representative winners.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "graphon_space").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from graphon_space.io import graphon_from_record

DATA = ROOT / "outputs" / "final" / "data"
DATA

In [ ]:
compare_triangle = pd.read_parquet(DATA / "compare_triangle_e035_t002.parquet")
compare_2star = pd.read_parquet(DATA / "compare_2star_e05_t029.parquet")
grid_triangle = pd.read_parquet(DATA / "entropy_grid_triangle.parquet")
grid_2star = pd.read_parquet(DATA / "entropy_grid_2star.parquet")

def target_summary(df):
    ok = df[df["success"]].copy()
    baseline = ok[ok["family"] == "symmetric-bipodal"]["entropy"].max() if (ok["family"] == "symmetric-bipodal").any() else np.nan
    best = ok.loc[ok["entropy"].idxmax()] if len(ok) else None
    return pd.Series({
        "successful candidates": len(ok),
        "best family": None if best is None else best["family"],
        "best entropy": np.nan if best is None else best["entropy"],
        "best class": None if best is None else best["symmetry_class"],
        "symmetric entropy": baseline,
        "best minus symmetric": np.nan if best is None or np.isnan(baseline) else best["entropy"] - baseline,
    })

pd.DataFrame({
    "triangle e=0.35 t=0.02": target_summary(compare_triangle),
    "2-star e=0.5 t=0.29": target_summary(compare_2star),
})

In [ ]:
def grid_gap(df):
    ok = df[df["success"]].copy()
    if ok.empty:
        return pd.DataFrame()
    best = ok.groupby(["target_e", "target_t"])["entropy"].max().rename("best_entropy")
    sym = ok[ok["family"] == "symmetric-bipodal"].groupby(["target_e", "target_t"])["entropy"].max().rename("symmetric_entropy")
    out = pd.concat([best, sym], axis=1).reset_index()
    out["gap"] = out["best_entropy"] - out["symmetric_entropy"]
    return out

tri_gap = grid_gap(grid_triangle)
star_gap = grid_gap(grid_2star)
display(tri_gap)
display(star_gap)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, df, title in [
    (axes[0], tri_gap, "Triangle grid: S_best - S_sym"),
    (axes[1], star_gap, "2-star grid: S_best - S_sym"),
]:
    if df.empty or df["gap"].dropna().empty:
        ax.text(0.5, 0.5, "No symmetric baseline in successful cells", ha="center", va="center")
        ax.set_axis_off()
    else:
        sc = ax.scatter(df["target_e"], df["target_t"], c=df["gap"], s=80, cmap="coolwarm")
        ax.set_xlabel("edge density e")
        ax.set_ylabel("t")
        ax.set_title(title)
        fig.colorbar(sc, ax=ax, label="entropy gap")
fig.tight_layout()
fig

In [ ]:
def best_graphon(df):
    ok = df[df["success"]].copy()
    row = ok.loc[ok["entropy"].idxmax()]
    return row, graphon_from_record(row.to_dict())

rows_graphons = [
    ("Triangle target winner", *best_graphon(compare_triangle)),
    ("2-star target winner", *best_graphon(compare_2star)),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (title, row, graphon) in zip(axes, rows_graphons):
    image = ax.imshow(graphon.matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(f"{title}\n{row['family']} / {row['symmetry_class']}")
    ax.set_xticks(range(graphon.k), [f"{x:.3f}" for x in graphon.sizes], rotation=45, ha="right")
    ax.set_yticks(range(graphon.k), [f"{x:.3f}" for x in graphon.sizes])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig